In [1]:
# ==========================================
# 1. PREPARACIÓN DEL ENTORNO
# ==========================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`


In [2]:
# Librerías necesarias
using ITensors
using ITensorMPS
using Plots
using LinearAlgebra

En este notebook se va simular la ecuación bioheat estacionaria mediante DMRG. En primer lugar se van a definir los parámetros del problema de contorno dado por:
		$$T''(x)-\sigma^2 T(x)=-f(x) \ , \ 0<x<L$$
		$$T'(0)=\beta T(0)-\gamma $$
		$$T(L)=\delta $$
		$$f(x)=F_0 e^{-\mu x}$$


In [3]:
# =============================================================================
# 1. PARÁMETROS DEL PROBLEMA
# =============================================================================
W = 1.0         
σ = 2.0         
β = 1.5         
γ = 0.5         
δ = 3.0         
F0 = 10.0       
μ = 1.0         

L_sites = 8     # ¡Súbelo a 20 cuando quieras!
N = 2^L_sites   
dx = W / N

println("--- Configuración QTT (LSB Puro) ---")
println("Número de qubits (Sitios): $L_sites")
println("Número de nodos (Dimensión N): $N")
println("-------------------------------------")

--- Configuración QTT (LSB Puro) ---
Número de qubits (Sitios): 8
Número de nodos (Dimensión N): 256
-------------------------------------


El primer paso es crear el MPS de fuentes. Existen dos opciones:
- Crear los MPS de fondo y extremos por separado y sumarlos, con la correspondiente compresión SVD.
- Crear un único MPS con el fondo y extremos simultáneamente. En este caso no es necesario realizar la compresión SVD.

**Opción 1: Construcción modular y suma con compresión SVD**

Esta primera aproximación aborda el problema dividiendo la física en dos componentes independientes: la generación de un fondo exponencial global y el escaneo de fronteras para aplicar correcciones delta locales en los extremos. Ambos Matrix Product States (MPS) se construyen por separado y se combinan finalmente en un único operador.

**¿Cómo funciona internamente?**

El procedimiento se divide en tres etapas secuenciales:

- Fondo Exponencial (mps_exponencial_lsb): Genera el perfil de decaimiento exponencial puro $f_i = A \cdot e^{-\mu \cdot dx \cdot i}$ en ordenación de bit menos significativo (LSB). Como el valor de cada sitio depende de un producto directo de factores independientes ($e^{-\mu \cdot dx \cdot 2^{i-1}}$), este estado se puede representar exactamente como un estado de producto sin entrelazamiento virtual, fijando la dimensión de enlace en $D = 1$.

- Corrección de Fronteras (mps_correccion_fronteras_lsb): Construye un autómata con dimensión de enlace $D = 2$ que actúa como un "escáner". Abre dos canales virtuales independientes: el primero sobrevive únicamente si todos los bits leídos aguas arriba son ceros (nodo izquierdo, $i=0$), y el segundo si todos son unos (nodo derecho, $i=2^L-1$). Al llegar al último sitio, el autómata descarga los valores de corrección $\Delta L$ y $\Delta R$ respectivamente.

- Suma y Compresión (add): Para unificar ambos efectos, se invoca la función add(mps_f, mps_bc; cutoff=1e-15). Internamente, ITensor realiza una suma directa de los espacios virtuales, generando un MPS intermedio con dimensión de enlace acumulada $D = 1 + 2 = 3$. Acto seguido, el entorno ejecuta automáticamente un barrido de ortogonalización y compresión mediante SVD a lo largo de todos los sitios para truncar el estado y eliminar redundancias basándose en el umbral de corte (cutoff) definido.

**Pros y Contras**

*Ventajas:*
- Alta modularidad: El código es muy limpio y fácil de mantener. Permite modificar, sustituir o eliminar la función del fondo exponencial (por ejemplo, cambiarlo por un fondo lineal o constante) sin necesidad de alterar en absoluto la lógica del autómata de las fronteras.
- Facilidad de depuración: Al tratarse de dos funciones independientes, es muy sencillo verificar la corrección numérica de cada MPS por separado antes de sumarlos.

*Desventajas:*
- Sobrecoste computacional: La operación de suma seguida de compresión exige calcular descomposiciones SVD o QR en cada uno de los $L$ sitios del sistema. Aunque para dimensiones pequeñas el coste es bajo, introduce un escalado temporal y un uso de CPU innecesarios.
- Pérdida de precisión matemática: La aproximación depende de un parámetro artificial (cutoff=1e-15). Aunque está cerca del límite de la precisión de doble flotante, introduce una microtruncación numérica en comparación con un método algebraicamente exacto.

In [4]:
# =============================================================================
# 2. MPS DE FUENTES (NATURALEZA LSB PURA) OPCIÓN 1
# =============================================================================

# Genera la exponencial f_i = A * exp(-μ * dx * i) en orden LSB (Sitio i = 2^(i-1))
function mps_exponencial_lsb(sites, A_amp, μ_exp, dx)
    L = length(sites)
    mps = MPS(sites)
    links = [Index(1, "Link,l=$i") for i in 1:(L-1)]
    
    for i in 1:L
        val_0 = 1.0
        val_1 = exp(-μ_exp * dx * 2^(i-1)) # LSB: El peso crece con i
        
        if i == 1
            A = ITensor(sites[1], links[1])
            A[sites[1]=>1, links[1]=>1] = A_amp * val_0
            A[sites[1]=>2, links[1]=>1] = A_amp * val_1
            mps[1] = A
        elseif i == L
            A = ITensor(links[L-1], sites[L])
            A[links[L-1]=>1, sites[L]=>1] = val_0
            A[links[L-1]=>1, sites[L]=>2] = val_1
            mps[L] = A
        else
            A = ITensor(links[i-1], sites[i], links[i])
            A[links[i-1]=>1, sites[i]=>1, links[i]=>1] = val_0
            A[links[i-1]=>1, sites[i]=>2, links[i]=>1] = val_1
            mps[i] = A
        end
    end
    return mps
end

# Escáner de fronteras: detecta todo ceros (nodo 1) o todo unos (nodo N) de izquierda a derecha
function mps_correccion_fronteras_lsb(sites, ΔL, ΔR)
    L = length(sites)
    mps = MPS(sites)
    links = [Index(2, "Link,l=$i") for i in 1:(L-1)]
    
    A1 = ITensor(sites[1], links[1])
    A1[sites[1]=>1, links[1]=>1] = 1.0 # Abre camino de ceros
    A1[sites[1]=>2, links[1]=>2] = 1.0 # Abre camino de unos
    mps[1] = A1
    
    for i in 2:(L-1)
        Ai = ITensor(links[i-1], sites[i], links[i])
        Ai[links[i-1]=>1, sites[i]=>1, links[i]=>1] = 1.0
        Ai[links[i-1]=>2, sites[i]=>2, links[i]=>2] = 1.0
        mps[i] = Ai
    end
    
    AL = ITensor(links[L-1], sites[L])
    AL[links[L-1]=>1, sites[L]=>1] = ΔL  # El camino de puros ceros deposita ΔL aquí
    AL[links[L-1]=>2, sites[L]=>2] = ΔR  # El camino de puros unos deposita ΔR aquí
    mps[L] = AL
    
    return mps
end

sites1 = siteinds("Qubit", L_sites)

ΔL = - (F0 / 2) - (γ / dx) - (-F0)
val_f_N = -F0 * exp(-μ * (N-1) * dx)
ΔR = (val_f_N - δ / dx^2) - val_f_N

mps_f = mps_exponencial_lsb(sites1, -F0, μ, dx)
mps_bc = mps_correccion_fronteras_lsb(sites1, ΔL, ΔR)
mps_objetivo1 = add(mps_f, mps_bc; cutoff=1e-15)

MPS
[1] ((dim=2|id=704|"Qubit,Site,n=1"), (dim=3|id=712|"Link,l=1"))
[2] ((dim=2|id=959|"Qubit,Site,n=2"), (dim=3|id=52|"Link,l=2"), (dim=3|id=712|"Link,l=1"))
[3] ((dim=2|id=171|"Qubit,Site,n=3"), (dim=3|id=581|"Link,l=3"), (dim=3|id=52|"Link,l=2"))
[4] ((dim=2|id=731|"Qubit,Site,n=4"), (dim=3|id=978|"Link,l=4"), (dim=3|id=581|"Link,l=3"))
[5] ((dim=2|id=216|"Qubit,Site,n=5"), (dim=3|id=320|"Link,l=5"), (dim=3|id=978|"Link,l=4"))
[6] ((dim=2|id=793|"Qubit,Site,n=6"), (dim=3|id=352|"Link,l=6"), (dim=3|id=320|"Link,l=5"))
[7] ((dim=2|id=111|"Qubit,Site,n=7"), (dim=2|id=685|"Link,l=7"), (dim=3|id=352|"Link,l=6"))
[8] ((dim=2|id=564|"Qubit,Site,n=8"), (dim=2|id=685|"Link,l=7"))


**Opción 2: Construcción manual integrada mediante un autómata de estados exacto**

Esta segunda aproximación consiste en diseñar y programar directamente el autómata de canales virtuales del MPS en una única función (genera_qtt_exponencial_cc). En lugar de construir los estados por separado y delegar en ITensor su posterior unificación y compresión, definimos de forma explícita los elementos de matriz fijando la dimensión de enlace óptima $D = 3$ desde el inicio.

**¿Cómo funciona internamente?**

El espacio virtual de la red se divide en tres canales con propósitos físicos bien definidos:
- Canal 1: Construye y propaga el fondo exponencial global.
- Canal 2 (Filtro Izquierdo): Detecta si el camino recorrido está compuesto por puros ceros ($|00\dots0\rangle$). Si aparece un solo bit $1$, el canal se destruye inmediatamente.
- Canal 3 (Filtro Derecho): Detecta si el camino recorrido está compuesto por puros unos ($|11\dots1\rangle$). Si aparece un solo bit $0$, el canal muere.

Matemáticamente, el estado local de cada sitio se define mediante tensores cuyos elementos dependen de la configuración del espacio físico del qubit (si el bit es $|0\rangle$ o $|1\rangle$), gobernados por los factores de escala $\lambda_i = e^{-\gamma \cdot 2^{i-1}}$:
- Sitio 1 (Vector fila $1 \times 3$): Inicializa el fondo exponencial y activa simultáneamente los canales de control izquierdo y derecho:$$W_1^{|0\rangle} = \begin{pmatrix} 1 & 1 & 0 \end{pmatrix}, \quad W_1^{|1\rangle} = \begin{pmatrix} \lambda_1 & 0 & 1 \end{pmatrix}$$
- Sitios intermedios $i$ (Matrices $3 \times 3$): El bit $|0\rangle$ propaga la exponencial y mantiene vivo el canal izquierdo de ceros (matando el derecho). El bit $|1\rangle$ escala el fondo exponencial por $\lambda_i$ y mantiene vivo el canal derecho de unos (matando el izquierdo):$$W_i^{|0\rangle} = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 0 \end{pmatrix}, \quad W_i^{|1\rangle} = \begin{pmatrix} \lambda_i & 0 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 1 \end{pmatrix}$$
- Sitio L (Vector columna $3 \times 1$): Actúa como el "cobrador o sumador final". Si el último bit es $|0\rangle$, cierra el fondo exponencial con la amplitud global $A_{\text{amp}}$ y absorbe la corrección $\Delta L$ (siempre que el camino viniera libre de unos). Si es $|1\rangle$, cierra la exponencial escalada y absorbe la corrección $\Delta R$ (siempre que el camino viniera libre de ceros):$$W_L^{|0\rangle} = \begin{pmatrix} A_{\text{amp}} \\ \Delta L \\ 0 \end{pmatrix}, \quad W_L^{|1\rangle} = \begin{pmatrix} A_{\text{amp}}\lambda_L \\ 0 \\ \Delta R \end{pmatrix}$$

**Pros y Contras**
*Ventajas:*
- Máxima eficiencia computacional: El coste algorítmico y el uso de memoria RAM escalan de forma estrictamente lineal ($\mathcal{O}(L)$) pura. No requiere operaciones ocultas de alojamiento temporal, ni llamadas a descomposiciones algebraicas (SVD o QR).
- Precisión matemática exacta: Al no depender de algoritmos de compresión SVD ni de un umbral de corte artificial (cutoff), el MPS resultante es algebraicamente exacto a nivel de precisión de máquina.
- Consumo de memoria óptimo: El pico de memoria durante la ejecución coincide exactamente con el tamaño del MPS final, procesando únicamente tensores locales de dimensiones muy reducidas ($3 \times 2 \times 3$).
*Desventajas:*
- Baja flexibilidad y modularidad: Toda la física del problema (el fondo y las fronteras) está acoplada dentro del mismo autómata. Si en el futuro se desea cambiar la naturaleza del fondo (por ejemplo, pasar de una exponencial a un polinomio), se debe rediseñar por completo la lógica analítica de las matrices de transferencia de los 3 canales.

🚀 **Ventaja clave para QTT:**

Este método integrado es el óptimo para fases de producción y escalado a gran escala ($L > 20$). Al eliminar por completo los cuellos de botella de memoria y CPU derivados de la compresión por SVD de la suma de estados, permite inicializar de manera instantánea vectores de fuentes altamente complejos en mallas hiperdensas.

In [5]:
function genera_qtt_exponencial_cc(L::Int, A_amp::Float64, γ::Float64, ΔL::Float64, ΔR::Float64)
    sites = siteinds("Qubit", L)
    links = [Index(3, "Link,l=$i") for i in 1:(L-1)]
    mps = MPS(sites)
    
    λ = [exp(-γ * 2^(i-1)) for i in 1:L]
    
    # --- SITIO 1 ---
    A1 = ITensor(sites[1], links[1])
    # Canal 1: Exponencial pura | Canal 2: Delta en 0 | Canal 3: Delta en 2^L - 1
    
    # Si bit = 0
    A1[sites[1]=>1, links[1]=>1] = 1.0
    A1[sites[1]=>1, links[1]=>2] = 1.0   # Activa el canal de la delta izquierda
    
    # Si bit = 1
    A1[sites[1]=>2, links[1]=>1] = λ[1]
    A1[sites[1]=>2, links[1]=>3] = 1.0   # Activa el canal de la delta derecha
    mps[1] = A1
    
    # --- SITIOS CENTRALES ---
    for i in 2:(L-1)
        Ai = ITensor(links[i-1], sites[i], links[i])
        
        # Si bit = 0 (propaga la exponencial y la delta izquierda; mata la derecha)
        Ai[links[i-1]=>1, sites[i]=>1, links[i]=>1] = 1.0
        Ai[links[i-1]=>2, sites[i]=>1, links[i]=>2] = 1.0
        
        # Si bit = 1 (propaga la exponencial y la delta derecha; mata la izquierda)
        Ai[links[i-1]=>1, sites[i]=>2, links[i]=>1] = λ[i]
        Ai[links[i-1]=>3, sites[i]=>2, links[i]=>3] = 1.0
        
        mps[i] = Ai
    end
    
    # --- SITIO L (Cobrador/Sumador final) ---
    AL = ITensor(links[L-1], sites[L])
    
    # Si bit = 0
    AL[links[L-1]=>1, sites[L]=>1] = A_amp         # Cierra exponencial
    AL[links[L-1]=>2, sites[L]=>1] = ΔL            # Cierra delta izquierda
    
    # Si bit = 1
    AL[links[L-1]=>1, sites[L]=>2] = A_amp * λ[L]  # Cierra exponencial
    AL[links[L-1]=>3, sites[L]=>2] = ΔR            # Cierra delta derecha
    
    mps[L] = AL
    return mps, sites
end

# Así debes llamarlo en tu script:
mps_objetivo2, sites2 = genera_qtt_exponencial_cc(L_sites, -F0, μ * dx, ΔL, ΔR)

(MPS
[1] ((dim=2|id=713|"Qubit,Site,n=1"), (dim=3|id=904|"Link,l=1"))
[2] ((dim=3|id=904|"Link,l=1"), (dim=2|id=117|"Qubit,Site,n=2"), (dim=3|id=846|"Link,l=2"))
[3] ((dim=3|id=846|"Link,l=2"), (dim=2|id=124|"Qubit,Site,n=3"), (dim=3|id=915|"Link,l=3"))
[4] ((dim=3|id=915|"Link,l=3"), (dim=2|id=859|"Qubit,Site,n=4"), (dim=3|id=905|"Link,l=4"))
[5] ((dim=3|id=905|"Link,l=4"), (dim=2|id=476|"Qubit,Site,n=5"), (dim=3|id=100|"Link,l=5"))
[6] ((dim=3|id=100|"Link,l=5"), (dim=2|id=80|"Qubit,Site,n=6"), (dim=3|id=457|"Link,l=6"))
[7] ((dim=3|id=457|"Link,l=6"), (dim=2|id=920|"Qubit,Site,n=7"), (dim=3|id=931|"Link,l=7"))
[8] ((dim=3|id=931|"Link,l=7"), (dim=2|id=554|"Qubit,Site,n=8"))
, Index{Int64}[(dim=2|id=713|"Qubit,Site,n=1"), (dim=2|id=117|"Qubit,Site,n=2"), (dim=2|id=124|"Qubit,Site,n=3"), (dim=2|id=859|"Qubit,Site,n=4"), (dim=2|id=476|"Qubit,Site,n=5"), (dim=2|id=80|"Qubit,Site,n=6"), (dim=2|id=920|"Qubit,Site,n=7"), (dim=2|id=554|"Qubit,Site,n=8")])

**Construcción del MPO Tridiagonal en Ordenación LSB con Contorno de Robin**

Esta función implementa la creación manual del Operador de Producto de Matrices (MPO) para una matriz tridiagonal de tamaño $N \times N$ (donde $N = 2^L$) utilizando la técnica QTT (Quantized Tensor Train) bajo una ordenación de bit menos significativo (LSB). 

**El Concepto Clave: Aritmética Binaria en Redes de Tensores**
En la ordenación LSB, el índice global de la matriz se codifica mediante la cadena de bits de izquierda a derecha, donde el primer qubit representa el peso numérico más bajo ($2^0$). Moverse a un elemento adyacente en la matriz real ($j \to j \pm 1$) equivale matemáticamente a sumar o restar 1 en binario.Para representar esto de forma exacta con un MPO de dimensión de enlace virtual fija $D = 4$, se diseñan 4 canales virtuales que actúan como un autómata finito:
- Canal 1 (q1 - Acarreo / Carry): Se activa cuando la interacción requiere sumar $1$ en binario (transición hacia la superdiagonal). Propaga un operador de bajada $S^-$ a través de los bits 1 (convirtiéndolos en 0) hasta que encuentra el primer bit 0, donde se resuelve aplicando un operador de subida $S^+$ multiplicado por la amplitud $b$.
- Canal 2 (q2 - Préstamo / Borrow): Se activa cuando se requiere restar $1$ en binario (transición hacia la subdiagonal). Propaga un operador de subida $S^+$ a través de los bits 0 (convirtiéndolos en 1) hasta que encuentra el primer bit 1, donde se resuelve aplicando un operador de bajada $S^-$ multiplicado por la amplitud $b$.
- Canal 3 (q3 - Resuelto / Acumulador): Es el canal neutro. Se encarga de arrastrar mediante la Identidad ($I$) los términos de la diagonal principal (parámetro $-a$) y todas las interacciones de acarreo o préstamo que ya han sido completamente resueltas en los qubits anteriores.
- Canal 4 (q4 - Filtro de Robin): Un canal completamente independiente que monitoriza el estado fundamental de los qubits. Está diseñado específicamente para aplicar la penalización de contorno $-c$ únicamente en el primer elemento global del sistema, correspondiente al estado de puros ceros $|00\dots0\rangle$.

**Estructura de Bloques del Autómata**

Matemáticamente, los tensores locales $W$ se construyen inyectando los operadores de espín apropiados ($I$, $P_0$, $S^+$, $S^-$) según el canal virtual de entrada y salida:
- Sitio 1 (Vector Fila $1 \times 4$)Inicializa los procesos aritméticos en el primer bit (el de peso $2^0$). Genera la diagonal local y los primeros hoppings de corto alcance, abriendo además las compuertas de los canales de acarreo, préstamo y Robin:$$W_1 = \begin{pmatrix} S^- & S^+ & -aI + bS^+ + bS^- & P_0 \end{pmatrix}$$
- Sitios Intermedios o Bulk (Matrices $4 \times 4$)Gobiernan la propagación de los acarreos binarios a lo largo de la cadena de espines. Nótese cómo los canales 1 y 2 se van vaciando progresivamente hacia el canal de "resuelto" (columna 3) tan pronto como encuentran el bit adecuado para cerrar la operación aritmética:$$W_i = \begin{pmatrix} 
S^- & 0 & bS^+ & 0 \\ 
0 & S^+ & bS^- & 0 \\ 
0 & 0 & I & 0 \\ 
0 & 0 & 0 & P_0 
\end{pmatrix}$$ 
- Sitio L (Vector Columna $4 \times 1$)Actúa como el cierre del autómata. Absorbe los acarreos o préstamos que hayan logrado llegar hasta el último bit del sistema y sella el canal de Robin aplicando la corrección final:$$W_L = \begin{pmatrix} bS^+ \\ bS^- \\ I \\ -cP_0 \end{pmatrix}$$

🎯 **Mecanismo de la Condición de Robin en QTT:**
Como el proyector $P_0 = |0\rangle\langle0|$ se exige consecutivamente a lo largo de todo el canal 4 desde el sitio 1 hasta el sitio $L$, el operador global resultante de esa rama es exclusivamente el proyector del primer nodo de la malla:$$\bigotimes_{i=1}^L P_0 = |00\dots0\rangle\langle00\dots0|$$Multiplicado por el factor $-c$ en el tensor de cierre $W_L$, el MPO modifica de forma exacta la esquina superior izquierda de la matriz global (elemento $(1,1)$), cumpliendo a la perfección la condición de contorno de Robin sin alterar el resto de la estructura tridiagonal. En formato QTT, esta sofisticada operación se logra manteniendo una eficiencia computacional óptima con un coste que escala solo linealmente con el número de qubits ($\mathcal{O}(L)$).

In [6]:
# =============================================================================
# 3. MPO TRIDIAGONAL CON CONDICIONES DE CONTORNO
# =============================================================================
function construir_mpo_manual_perfecto_lsb(sites, a::Float64, b::Float64, c::Float64)

    L = length(sites)

    MPO_manual = MPO(sites)
    links = [Index(4, "Link,l=$l") for l in 1:(L-1)]

    Id = [1.0 0.0; 0.0 1.0]
    P0 = [1.0 0.0; 0.0 0.0]

    Sm = [0.0 1.0; 0.0 0.0]
    Sp = [0.0 0.0; 1.0 0.0]


    # --------------------------------------------------
    # SITIO 1
    # --------------------------------------------------

    W1 = ITensor(sites[1], prime(sites[1]), links[1])

    for s in 1:2, sp in 1:2

        # q1 = acarreo
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>1] =
            Sm[s,sp]

        # q2 = préstamo
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>2] =
            Sp[s,sp]

        # q3 = resuelto
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>3] =
            -a*Id[s,sp] + b*Sp[s,sp] + b*Sm[s,sp]

        # q4 = Robin
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>4] =
            P0[s,sp]

    end

    MPO_manual[1] = W1

    # --------------------------------------------------
    # BULK
    # --------------------------------------------------

    for i in 2:(L-1)

        W = ITensor(
            links[i-1],
            sites[i],
            prime(sites[i]),
            links[i]
        )

        for s in 1:2, sp in 1:2

            # q1 -> q1
            W[links[i-1]=>1,
              sites[i]=>s,
              prime(sites[i])=>sp,
              links[i]=>1] = Sm[s,sp]

            # q1 -> q3
            W[links[i-1]=>1,
              sites[i]=>s,
              prime(sites[i])=>sp,
              links[i]=>3] = b*Sp[s,sp]

            # q2 -> q2
            W[links[i-1]=>2,
              sites[i]=>s,
              prime(sites[i])=>sp,
              links[i]=>2] = Sp[s,sp]

            # q2 -> q3
            W[links[i-1]=>2,
              sites[i]=>s,
              prime(sites[i])=>sp,
              links[i]=>3] = b*Sm[s,sp]

            # q3 -> q3
            W[links[i-1]=>3,
              sites[i]=>s,
              prime(sites[i])=>sp,
              links[i]=>3] = Id[s,sp]

            # q4 -> q4
            W[links[i-1]=>4,
              sites[i]=>s,
              prime(sites[i])=>sp,
              links[i]=>4] = P0[s,sp]

        end

        MPO_manual[i] = W

    end

    # --------------------------------------------------
    # CIERRE
    # --------------------------------------------------

    WN = ITensor(
        links[L-1],
        sites[L],
        prime(sites[L])
    )

    for s in 1:2, sp in 1:2

        WN[links[L-1]=>1,
           sites[L]=>s,
           prime(sites[L])=>sp] = b*Sp[s,sp]

        WN[links[L-1]=>2,
           sites[L]=>s,
           prime(sites[L])=>sp] = b*Sm[s,sp]

        WN[links[L-1]=>3,
           sites[L]=>s,
           prime(sites[L])=>sp] = Id[s,sp]

        WN[links[L-1]=>4,
           sites[L]=>s,
           prime(sites[L])=>sp] = -c*P0[s,sp]

    end

    MPO_manual[L] = WN

    return MPO_manual

end

val_a = 2.0 / dx^2 + σ^2
val_b = 1.0 / dx^2
A_11_real = -(1.0/dx^2 + β/dx + σ^2/2)
val_c = -val_a - A_11_real

# Construimos el MPO para los sites de la opción 1 y la opción 2 del MPS de fuentes
MPO_tridiag1 = construir_mpo_manual_perfecto_lsb(sites1, val_a, val_b, val_c)
MPO_tridiag2 = construir_mpo_manual_perfecto_lsb(sites2, val_a, val_b, val_c)

MPO
[1] ((dim=2|id=713|"Qubit,Site,n=1"), (dim=2|id=713|"Qubit,Site,n=1")', (dim=4|id=681|"Link,l=1"))
[2] ((dim=4|id=681|"Link,l=1"), (dim=2|id=117|"Qubit,Site,n=2"), (dim=2|id=117|"Qubit,Site,n=2")', (dim=4|id=733|"Link,l=2"))
[3] ((dim=4|id=733|"Link,l=2"), (dim=2|id=124|"Qubit,Site,n=3"), (dim=2|id=124|"Qubit,Site,n=3")', (dim=4|id=171|"Link,l=3"))
[4] ((dim=4|id=171|"Link,l=3"), (dim=2|id=859|"Qubit,Site,n=4"), (dim=2|id=859|"Qubit,Site,n=4")', (dim=4|id=108|"Link,l=4"))
[5] ((dim=4|id=108|"Link,l=4"), (dim=2|id=476|"Qubit,Site,n=5"), (dim=2|id=476|"Qubit,Site,n=5")', (dim=4|id=184|"Link,l=5"))
[6] ((dim=4|id=184|"Link,l=5"), (dim=2|id=80|"Qubit,Site,n=6"), (dim=2|id=80|"Qubit,Site,n=6")', (dim=4|id=770|"Link,l=6"))
[7] ((dim=4|id=770|"Link,l=6"), (dim=2|id=920|"Qubit,Site,n=7"), (dim=2|id=920|"Qubit,Site,n=7")', (dim=4|id=379|"Link,l=7"))
[8] ((dim=4|id=379|"Link,l=7"), (dim=2|id=554|"Qubit,Site,n=8"), (dim=2|id=554|"Qubit,Site,n=8")')


In [8]:
# =============================================================================
# 4. RESOLUCIÓN VARIACIONAL VIA LINSOLVE EN EL CASO 1 DEL MPS DE FUENTES
# =============================================================================
println("Ejecutando linsolve...")
mps_unos1 = random_mps(sites1; linkdims=4)
T_mps1 = linsolve(MPO_tridiag1, mps_objetivo1, mps_unos1; cutoff=1e-12, maxdim=40, nsweeps=10)
mps_unos2 = random_mps(sites2; linkdims=4)
T_mps2 = linsolve(MPO_tridiag2, mps_objetivo2, mps_unos2; cutoff=1e-12, maxdim=40, nsweeps=10)

# =============================================================================
# 5. VALIDACIÓN EN LSB PURO (SIN REVERSE)
# =============================================================================
if L_sites <= 8
    A_mat = zeros(N, N); b_vec = zeros(N); f = [F0 * exp(-μ * i * dx) for i in 0:N-1]
    A_mat[1, 1] = -(1/dx^2 + β/dx + σ^2/2); A_mat[1, 2] = 1/dx^2; b_vec[1] = -f[1]/2 - γ/dx
    for i in 2:N-1
        A_mat[i, i-1] = 1/dx^2; A_mat[i, i] = -(2/dx^2 + σ^2); A_mat[i, i+1] = 1/dx^2; b_vec[i] = -f[i]
    end
    A_mat[N, N-1] = 1/dx^2; A_mat[N, N] = -(2/dx^2 + σ^2); b_vec[N] = -f[N] - δ/dx^2
    T_clasico = SymTridiagonal(diag(A_mat), diag(A_mat, 1)) \ b_vec

    T_resultado_dmrg1 = zeros(N)
    for idx in 1:N
        # ¡CONVENCIÓN INICIAL RECUPERADA!: LSB directo, sin reverse
        bits = digits(idx - 1, base=2, pad=L_sites)
        
        psi = T_mps1[1] * state(sites1[1], bits[1] + 1)
        for s in 2:L_sites
            psi = psi * T_mps1[s] * state(sites1[s], bits[s] + 1)
        end
        T_resultado_dmrg1[idx] = scalar(psi)
    end
    T_resultado_dmrg2 = zeros(N)
    for idx in 1:N
        # ¡CONVENCIÓN INICIAL RECUPERADA!: LSB directo, sin reverse
        bits = digits(idx - 1, base=2, pad=L_sites)
        
        psi = T_mps2[1] * state(sites2[1], bits[1] + 1)
        for s in 2:L_sites
            psi = psi * T_mps2[s] * state(sites2[s], bits[s] + 1)
        end
        T_resultado_dmrg2[idx] = scalar(psi)
    end

    println("\n==================================================")
    println("Método Clásico: ", round.(T_clasico[1:5], digits=5))
    println("QTT-DMRG Puro 1: ", round.(T_resultado_dmrg1[1:5], digits=5))     
    println("QTT-DMRG Puro 2: ", round.(T_resultado_dmrg2[1:5], digits=5))
    println("Error de norma Euclídea Absoluta 1: ", norm(T_clasico - T_resultado_dmrg1))
    println("Error de norma Euclídea Absoluta 2: ", norm(T_clasico - T_resultado_dmrg2))
    println("==================================================")
end

Ejecutando linsolve...

Método Clásico: [1.4156, 1.42191, 1.42815, 1.43433, 1.44045]
QTT-DMRG Puro 1: [1.4156, 1.42191, 1.42815, 1.43433, 1.44045]
QTT-DMRG Puro 2: [1.4156, 1.42191, 1.42815, 1.43433, 1.44045]
Error de norma Euclídea Absoluta 1: 3.740183498817747e-9
Error de norma Euclídea Absoluta 2: 7.456509574626224e-9


Al usar add en la Opción 1, ITensor realiza una ortogonalización QR/SVD previa que puede dejar los índices del MPS en un gauge (calibración de medidor) ligeramente más favorable para las condiciones iniciales del solver de Krylov interno de linsolve. No obstante, ambos errores están en el mismo orden de magnitud.